# Aleph quickstart

Use an existing notebook environment. No package installation is needed. Supply your existing key through the environment; never put it in the notebook. The example sends one short synthetic request. Keep real research outputs in storage appropriate to their classification.


In [ ]:
import os, json, urllib.request, datetime
from pathlib import Path
base = "https://inference.vulcan.alliancecan.ca/v1"
key = os.environ["ALEPH_API_KEY"]
headers = {"Authorization": "Bearer " + key, "Content-Type": "application/json"}


In [ ]:
request = urllib.request.Request(base + "/models", headers=headers)
with urllib.request.urlopen(request, timeout=30) as response:
    models = json.load(response)
[item["id"] for item in models["data"]]


Choose a model with the required capabilities and its own context/output limits. The request below leaves sampling and effort choices explicit. Temperature zero does not guarantee reproducibility.


In [ ]:
body = {"model": "qwen38-27b", "messages": [{"role": "user", "content": "Explain a confidence interval in two sentences."}], "reasoning_effort": "none", "temperature": 0, "max_tokens": 128}
request = urllib.request.Request(base + "/chat/completions", headers=headers, data=json.dumps(body).encode())
with urllib.request.urlopen(request, timeout=600) as response:
    result = json.load(response)
print(result["choices"][0]["message"]["content"])
print(result.get("usage", {}))


Save the request parameters, response and date in your scratch workspace. The key is excluded. Move important cleaned results to project storage; scratch is temporary. This cell creates a new file and refuses to overwrite an existing one.


In [ ]:
record = {"timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "endpoint": base, "request": body, "response": result}
output = Path(os.environ["SCRATCH"]) / "aleph-quickstart-result.json"
with output.open("x") as destination:
    json.dump(record, destination, indent=2)
print("Saved", output)
